In [2]:
import pyspark
from pyspark.sql import SparkSession

Question#1

In [7]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

25/03/05 01:25:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [8]:
spark.version

'3.5.5'

Question#2

In [15]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-05 01:29:04--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.209.126, 54.230.209.140, 54.230.209.72, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.209.126|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  90.6MB/s    in 0.7s    

2025-03-05 01:29:05 (90.6 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [16]:
df = spark.read \
    .option("header", "true") \
    .parquet("yellow_tripdata_2024-10.parquet")


In [ ]:
_partitioneddf = df.repartition(4)

In [18]:
print(df_partitioned.rdd.getNumPartitions())

4


In [19]:
df_partitioned.write.mode("overwrite").parquet("partitioned/")

In [21]:
import os
os.system("du -sh partitioned/part-00000-e96062c5-af33-4cb6-86c5-33e31dcff481-c000.snappy.parquet")

23M	partitioned/part-00000-e96062c5-af33-4cb6-86c5-33e31dcff481-c000.snappy.parquet


0

Question#3

In [24]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-10-07 16:40:43|  2024-10-07 18:10:56|              1|         14.8|        99|                 N|         127|         225|           1|       47.5|  0.0|    0.5|       0.

In [26]:
trip_counts = df_partitioned.filter((df_partitioned.tpep_pickup_datetime >= '2024-10-15') & (df_partitioned.tpep_pickup_datetime < '2024-10-16')).count()

print(f"Number of trips on the 15th of October: {trip_counts}")

Number of trips on the 15th of October: 128893


Question#4

In [33]:
from pyspark.sql.functions import unix_timestamp

df_with_diff_hours = df_partitioned.withColumn(
    "time_diff_hours", 
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600
)

df_with_diff_hours.agg({"time_diff_hours": "max"}).collect()

[Row(max(time_diff_hours)=162.61777777777777)]

Question#6

In [35]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-05 01:54:20--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.209.126, 54.230.209.200, 54.230.209.72, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.209.126|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.002s  

2025-03-05 01:54:20 (5.98 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [46]:
df_least_frequent_pulocation = df_partitioned.groupBy("PULocationID").count()

In [41]:
lookup_table = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

In [47]:
lookup_table.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [48]:
df_join = df_least_frequent_pulocation.join(lookup_table, df_least_frequent_pulocation["PULocationID"] == lookup_table["LocationID"], how="left").orderBy("count", ascending=True)

In [49]:
df_join.show()

+------------+-----+----------+-------------+--------------------+------------+
|PULocationID|count|LocationID|      Borough|                Zone|service_zone|
+------------+-----+----------+-------------+--------------------+------------+
|         105|    1|       105|    Manhattan|Governor's Island...| Yellow Zone|
|           5|    2|         5|Staten Island|       Arden Heights|   Boro Zone|
|         199|    2|       199|        Bronx|       Rikers Island|   Boro Zone|
|           2|    3|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         111|    3|       111|     Brooklyn| Green-Wood Cemetery|   Boro Zone|
|          44|    4|        44|Staten Island|Charleston/Totten...|   Boro Zone|
|          84|    4|        84|Staten Island|Eltingville/Annad...|   Boro Zone|
|         245|    4|       245|Staten Island|       West Brighton|   Boro Zone|
|         204|    4|       204|Staten Island|   Rossville/Woodrow|   Boro Zone|
|         187|    4|       187|Staten Is